# Get from LHE

Reads the LHE files in order to get the known valid points, getting the the list of important parameters

In [1]:

import os
import glob
import re
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR = "valid_points_lhe"    # your folder with *.lha files
OUTPUT_CSV = "valid_points.csv"

# ─────────────────────────────────────────────────────────────────────────────
# REGEX TO CATCH LINES OF THE FORM:
#   <code>   <value>   # <comment>
# e.g. "   18   4.40990900e+00    # m_12^2"
# ─────────────────────────────────────────────────────────────────────────────
line_re = re.compile(r'^\s*(\d+)\s+([+-]?\d*\.\d+(?:[eE][+-]?\d+)?)\s*#\s*(.+)$')

# ─────────────────────────────────────────────────────────────────────────────
# MAPPING FROM CODE → COLUMN NAME
# ─────────────────────────────────────────────────────────────────────────────
MINPAR_MAP = {
    3:  "tan_beta",
    11: "lambda1",
    12: "lambda2",
    13: "lambda3",
    14: "lambda4",
    15: "lambda5",
    16: "lambda6",
    17: "lambda7",
    18: "m12_2",
    20: "sin_ba",
    21: "cos_ba",
    24: "yukawa_type",
}
MASS_MAP = {
    25: "Mh1",
    35: "Mh2",
    36: "Mh3",
}

# ─────────────────────────────────────────────────────────────────────────────
# LOOP OVER FILES
# ─────────────────────────────────────────────────────────────────────────────
records = []

for path in sorted(glob.glob(os.path.join(DATA_DIR, "*.lha"))):
    rec = {"file": os.path.basename(path)}
    block = None

    with open(path) as f:
        for line in f:
            up = line.strip().upper()
            # detect block entry
            if up.startswith("BLOCK MINPAR"):
                block = "MINPAR"
                continue
            elif up.startswith("BLOCK MASS"):
                block = "MASS"
                continue
            elif up.startswith("BLOCK"):
                block = None
                continue

            if block in ("MINPAR", "MASS"):
                m = line_re.match(line)
                if not m:
                    continue
                code = int(m.group(1))
                val  = float(m.group(2))
                # choose the right map
                if block == "MINPAR" and code in MINPAR_MAP:
                    rec[MINPAR_MAP[code]] = val
                elif block == "MASS"  and code in MASS_MAP:
                    rec[MASS_MAP[code]] = val

    records.append(rec)

# ─────────────────────────────────────────────────────────────────────────────
# BUILD DATAFRAME & SAVE
# ─────────────────────────────────────────────────────────────────────────────
df = pd.DataFrame(records)

# fill any missing columns with NaN
all_cols = ["file"] + list(MINPAR_MAP.values()) + list(MASS_MAP.values())
df = df.reindex(columns=all_cols)

df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(df)} valid‐point entries to {OUTPUT_CSV}")


Wrote 23 valid‐point entries to valid_points.csv
